# UCLA Protest Image Classification

This notebook implements a pipeline from data loading to model evaluation and figure generation. It uses the UCLA Protest Image Dataset.

The pipeline compares three tiers of model complexity:
- **Tier 1:** Frozen random CNN features + Logistic Regression
- **Tier 2:** Fine‑tuned CNN (with/without data augmentation)
- **Tier 3:** Small Vision Transformer (ViT) trained from scratch

We measure compute cost (trainable parameters and wall‑clock time) and performance (F1, accuracy) to analyse the trade‑off.

To run the notebook, place it alongside 'annot_train.txt', 'annot_test.txt' and the img folder containing the train and test image folders.

## 1. Imports & Configuration

We use TensorFlow/Keras for deep learning, scikit‑learn for the logistic regression baseline, and standard libraries for data handling and plotting. All parameters are centralised in a `Config` dataclass.

In [ ]:
import os
import json
import time
import logging
import argparse
from dataclasses import dataclass
from typing import Dict, Any, Optional, Callable, Tuple

import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
f1_score, confusion_matrix, precision_score,
recall_score, accuracy_score
)
import matplotlib
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)


In [ ]:
@dataclass
class Config:
    DATA_DIR: str = "."
    IMG_SUBDIR: str = "img"
    OUT_DIR: str = "."
    CACHE_DIR: str = ""
    RESULTS_DIR: str = ""
    FIG_DIR: str = ""

    IMG_SIZE: int = 96
    SEED: int = 43
    DATA_SEED: int = 43
    # Close to the full corpus
    N_TRAIN: int = 29349
    N_VAL: int = 3261
    N_TEST: int = 8152
    POS_RATE: float = 0.286

    EPOCHS: int = 8
    BATCH_SIZE: int = 32
    LEARNING_RATE: float = 1e-3

    VIT_PATCH: int = 16
    VIT_DIM: int = 64
    VIT_HEADS: int = 4
    VIT_DEPTH: int = 3
    VIT_MLP: int = 128

    def __post_init__(self):
        self.CACHE_DIR = os.path.join(self.OUT_DIR, "data_cache")
        self.RESULTS_DIR = os.path.join(self.OUT_DIR, "results_cache")
        self.FIG_DIR = os.path.join(self.OUT_DIR, "figures")
        for d in [self.CACHE_DIR, self.RESULTS_DIR, self.FIG_DIR]:
            os.makedirs(d, exist_ok=True)



## 2. Data Loading

The dataset consists of images and annotation files. We load the annotations, subsample them to a fixed size while preserving the positive class ratio (≈28.6%), and load the corresponding images.

In [ ]:
def validate_data_layout(cfg: "Config") -> None:
    """Fail if required dataset files/folders are missing."""
    root = os.path.abspath(cfg.DATA_DIR)
    expected = {
        "annot_train.txt": os.path.join(root, "annot_train.txt"),
        "annot_test.txt": os.path.join(root, "annot_test.txt"),
        f"{cfg.IMG_SUBDIR}/train/": os.path.join(root, cfg.IMG_SUBDIR, "train"),
        f"{cfg.IMG_SUBDIR}/test/": os.path.join(root, cfg.IMG_SUBDIR, "test"),
    }
    missing = [label for label, path in expected.items() if not os.path.exists(path)]
    if missing:
        listing = "\n".join(f"  - {label}" for label in missing)
        raise FileNotFoundError(
            f"Dataset not found where expected under DATA_DIR='{root}'.\n"
            f"Missing:\n{listing}\n\n"
            "Expected layout:\n"
            "  <DATA_DIR>/annot_train.txt\n"
            "  <DATA_DIR>/annot_test.txt\n"
            f"  <DATA_DIR>/{cfg.IMG_SUBDIR}/train/*.jpg\n"
            f"  <DATA_DIR>/{cfg.IMG_SUBDIR}/test/*.jpg\n\n"
            "Fix: set Config.DATA_DIR to the dataset root."
        )


def validate_annotation_images(
    df: pd.DataFrame, img_dir: str, split: str
) -> None:
    """Verify every annotated image exists before model training begins."""
    missing = []
    for fn in df["fname"].astype(str):
        p = os.path.join(img_dir, fn)
        if not os.path.isfile(p):
            alt = os.path.join(os.path.dirname(img_dir), fn)
            if os.path.isfile(alt):
                continue
            missing.append(fn)

    if missing:
        preview = "\n".join(f"  - {x}" for x in missing[:10])
        more = "" if len(missing) <= 10 else f"\n  ... and {len(missing) - 10} more"
        raise FileNotFoundError(
            f"{split} annotation references {len(missing)} missing image(s).\n"
            f"Image directory checked: {os.path.abspath(img_dir)}\n"
            f"First missing files:\n{preview}{more}\n\n"
            "The annotation file and image folders are not consistent."
        )
    logger.info("%s image completeness check passed (%d annotations).", split, len(df))


In [ ]:
def build_dataset(cfg: Config) -> Dict[str, Any]:
    logger.info("Loading dataset from %s", os.path.abspath(cfg.DATA_DIR))
    validate_data_layout(cfg)

    train_full = load_annotations("train", cfg.DATA_DIR)
    test_full = load_annotations("test", cfg.DATA_DIR)

    train_img_dir = os.path.join(cfg.DATA_DIR, cfg.IMG_SUBDIR, "train")
    test_img_dir = os.path.join(cfg.DATA_DIR, cfg.IMG_SUBDIR, "test")

    # One-time completeness check on the full annotated corpus.
    validate_annotation_images(train_full, train_img_dir, "train")
    validate_annotation_images(test_full, test_img_dir, "test")

    train_sub = stratified_subsample(
        train_full, cfg.N_TRAIN + cfg.N_VAL, cfg.DATA_SEED, cfg.POS_RATE
    )
    pos = train_sub[train_sub["protest"] == 1]
    neg = train_sub[train_sub["protest"] == 0]
    n_val_pos = int(round(cfg.N_VAL * cfg.POS_RATE))
    n_val_neg = cfg.N_VAL - n_val_pos

    if len(pos) < n_val_pos or len(neg) < n_val_neg:
        raise ValueError(
            "Training subsample does not contain enough examples for "
            "the requested validation split."
        )

    val_df = pd.concat([
        pos.iloc[:n_val_pos],
        neg.iloc[:n_val_neg],
    ]).sample(frac=1.0, random_state=cfg.DATA_SEED + 3)

    train_df = train_sub.drop(val_df.index).sample(
        frac=1.0, random_state=cfg.DATA_SEED + 4
    )
    test_df = stratified_subsample(
        test_full, cfg.N_TEST, cfg.DATA_SEED + 5, cfg.POS_RATE
    )

    Xtr, train_df = load_images(train_df, train_img_dir, cfg.IMG_SIZE)
    Xva, val_df = load_images(val_df, train_img_dir, cfg.IMG_SIZE)
    Xte, test_df = load_images(test_df, test_img_dir, cfg.IMG_SIZE)

    return {
        "Xtr": Xtr, "ytr": train_df["protest"].values.astype(np.int64),
        "Xva": Xva, "yva": val_df["protest"].values.astype(np.int64),
        "Xte": Xte, "yte": test_df["protest"].values.astype(np.int64),
        "source": "UCLA Protest Image Dataset",
    }


In [ ]:
def load_annotations(split: str, data_dir: str) -> pd.DataFrame:
    fname = "annot_train.txt" if split == "train" else "annot_test.txt"
    path = os.path.join(data_dir, fname)
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Annotation file not found: {path}")
    df = pd.read_csv(path, sep="\t", na_values=["-"])
    df.columns = [c.strip() for c in df.columns]
    required = {"fname", "protest"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{fname} is missing required columns: {sorted(missing)}")
    return df

def stratified_subsample(df: pd.DataFrame, n: int, seed: int, pos_rate: float) -> pd.DataFrame:
    pos = df[df["protest"] == 1]
    neg = df[df["protest"] == 0]
    n_pos = int(round(n * pos_rate))
    n_neg = n - n_pos
    # Guard against insufficient samples
    if len(pos) < n_pos:
        n_pos = len(pos)
        n_neg = n - n_pos
    if len(neg) < n_neg:
        n_neg = len(neg)
        n_pos = n - n_neg
    pos_sampled = pos.sample(n=n_pos, random_state=seed)
    neg_sampled = neg.sample(n=n_neg, random_state=seed+1)
    return pd.concat([pos_sampled, neg_sampled]).sample(frac=1.0, random_state=seed+2)

def load_images(df: pd.DataFrame, img_dir: str, img_size: int):
    # Fail on an unreadable file
    images = []
    for fname in df["fname"].values:
        path = os.path.join(img_dir, fname)
        try:
            img = Image.open(path).convert("RGB")
            img = img.resize((img_size, img_size), Image.BILINEAR)
            images.append(np.array(img) / 255.0)
        except Exception as e:
            raise OSError(
                f"Image file exists but could not be read/decoded: {path} ({e}). "
                f"This file is corrupted or not a valid image; remove it or its "
                f"annotation row before rerunning."
            ) from e
    return np.stack(images, axis=0).astype(np.float32), df

In [ ]:
def _dataset_fingerprint(cfg: Config) -> Dict[str, Any]:
    """Every Config field that changes what build_dataset() produces.
    The cache is only valid if all of these still match"""
    return {
        "img_size": cfg.IMG_SIZE,
        "n_train": cfg.N_TRAIN,
        "n_val": cfg.N_VAL,
        "n_test": cfg.N_TEST,
        "pos_rate": cfg.POS_RATE,
        "data_seed": cfg.DATA_SEED,
    }


def get_dataset(cfg: Config, force_rebuild: bool = False) -> Dict[str, Any]:
    cache_npz = os.path.join(cfg.CACHE_DIR, "dataset.npz")
    cache_meta = os.path.join(cfg.CACHE_DIR, "dataset_meta.json")
    fingerprint = _dataset_fingerprint(cfg)

    if not force_rebuild and os.path.isfile(cache_npz) and os.path.isfile(cache_meta):
        try:
            with open(cache_meta) as f:
                meta = json.load(f)
            if all(meta.get(k) == v for k, v in fingerprint.items()):
                logger.info("Loading cached dataset")
                d = np.load(cache_npz)
                cached = {
                    k: d[k] for k in ["Xtr", "ytr", "Xva", "yva", "Xte", "yte"]
                }
                cached["source"] = "UCLA Protest Image Dataset"
                return cached
            else:
                logger.info(
                    "Cached dataset config does not match current Config "
                    "(cached=%s, current=%s); rebuilding.",
                    {k: meta.get(k) for k in fingerprint}, fingerprint,
                )
        except Exception as exc:
            logger.warning("Could not load dataset cache; rebuilding: %s", exc)

    data = build_dataset(cfg)
    np.savez_compressed(
        cache_npz,
        Xtr=data["Xtr"], ytr=data["ytr"],
        Xva=data["Xva"], yva=data["yva"],
        Xte=data["Xte"], yte=data["yte"],
    )
    with open(cache_meta, "w") as f:
        json.dump({
            **fingerprint,
            "n_train_actual": len(data["ytr"]),
            "n_val_actual": len(data["yva"]),
            "n_test_actual": len(data["yte"]),
            "train_pos_rate_actual": float(np.mean(data["ytr"])),
        }, f, indent=2)
    return data


## 3. Evaluation Helpers

We compute standard classification metrics (F1, precision, recall, accuracy) and a confusion matrix. The `save_result` function stores results as JSON for later analysis and figure generation.

In [ ]:
def evaluate(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_pred = (np.asarray(y_prob).ravel() >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return {
        "f1_binary_positive": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_positive": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "recall_positive": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "confusion_matrix": cm.tolist(),
        "n_test": int(len(y_true)),
        "test_pos_rate": float(np.mean(y_true)),
    }

def save_result(tag, record, results_dir):
    path = os.path.join(results_dir, f"{tag}.json")
    with open(path, "w") as f:
        json.dump(record, f, indent=2)
    logger.info("Saved result to %s", path)


## 4. Model Builders (TensorFlow/Keras)

We define three model architectures:

1. Frozen backbone: a small CNN with all layers frozen, used as a fixed feature extractor.
2. Fine‑tuned CNN: the same architecture but trainable, with optional data augmentation.
3. Vision Transformer (ViT): a minimal transformer from scratch (patch embedding, positional encoding, multi‑head attention).

The `count_*_params` functions help us compare compute costs.

In [ ]:
def set_seed(seed):
    """Set global random seed for reproducible model initialisation."""
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

def build_frozen_backbone(img_size=96, seed=43):
    """Build an untrained 3-layer CNN used as a frozen feature extractor (Tier 1)."""
    set_seed(seed)
    inp = tf.keras.layers.Input((img_size, img_size, 3))
    x = tf.keras.layers.Conv2D(16, 3, strides=2, padding="same", activation="relu")(inp)
    x = tf.keras.layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(x)
    x = tf.keras.layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    model = tf.keras.Model(inp, x, name="frozen_backbone")
    for layer in model.layers:
        layer.trainable = False
    return model

def build_finetune_cnn(img_size=96, seed=43, augment=False):
    """Build a small CNN trained end-to-end, optionally with data augmentation (Tier 2/2b)."""
    set_seed(seed)
    inp = tf.keras.layers.Input((img_size, img_size, 3))
    x = inp
    if augment:
        x = tf.keras.layers.RandomFlip("horizontal")(x)
        x = tf.keras.layers.RandomRotation(0.06)(x)
        x = tf.keras.layers.RandomContrast(0.15)(x)
    for filters, stride in [(16, 2), (32, 2), (64, 2)]:
        x = tf.keras.layers.Conv2D(filters, 3, strides=stride, padding="same")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(32, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inp, out, name="finetune_cnn" + ("_aug" if augment else ""))

class AddPositionEmbs(tf.keras.layers.Layer):
    """Learned positional embedding added to ViT patch tokens."""

    def build(self, input_shape):
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, input_shape[1], input_shape[2]),
            initializer=tf.random_normal_initializer(stddev=0.02),
            trainable=True,
        )

    def call(self, inputs):
        return inputs + self.pos_embed


def build_vit_scratch(img_size=96, patch=16, dim=64, heads=4, depth=3, mlp=128, seed=43):
    """Build a Vision Transformer trained from random initialisation (Tier 3)."""
    set_seed(seed)
    assert img_size % patch == 0
    num_patches = (img_size // patch) ** 2
    inp = tf.keras.layers.Input((img_size, img_size, 3))
    x = tf.keras.layers.Conv2D(dim, kernel_size=patch, strides=patch, name="patch_embed")(inp)
    x = tf.keras.layers.Reshape((num_patches, dim))(x)
    x = AddPositionEmbs()(x)
    for i in range(depth):
        a = tf.keras.layers.LayerNormalization(name=f"ln1_{i}")(x)
        a = tf.keras.layers.MultiHeadAttention(num_heads=heads, key_dim=dim // heads, name=f"mha_{i}")(a, a)
        x = tf.keras.layers.Add()([x, a])
        f = tf.keras.layers.LayerNormalization(name=f"ln2_{i}")(x)
        f = tf.keras.layers.Dense(mlp, activation="gelu", name=f"mlp1_{i}")(f)
        f = tf.keras.layers.Dense(dim, name=f"mlp2_{i}")(f)
        x = tf.keras.layers.Add()([x, f])
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    out = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inp, out, name="vit_scratch")

def count_trainable_params(model):
    """Return the number of trainable parameters in a Keras model."""
    return int(sum(tf.size(w).numpy() for w in model.trainable_weights))

def count_nontrainable_params(model):
    """Return the number of frozen (non-trainable) parameters in a Keras model."""
    return int(sum(tf.size(w).numpy() for w in model.non_trainable_weights))


## 5. Generic Training Loop (DRY)

To avoid repetition, we implement a single `train_and_evaluate` function that accepts any model builder, trains it, evaluates on the validation and test sets, and returns a structured result dictionary. This is used for both Tier 2 and Tier 3.

In [ ]:
def train_and_evaluate(
    model_builder: Callable,
    data: Dict[str, Any],
    cfg: Config,
    tag: str,
    description: str,
    builder_kwargs: Optional[Dict] = None,
) -> Dict[str, Any]:
    """Train one model end-to-end, time it, and score it on val/test sets."""
    builder_kwargs = builder_kwargs or {}
    Xtr, ytr = data["Xtr"], data["ytr"]
    Xva, yva = data["Xva"], data["yva"]
    Xte, yte = data["Xte"], data["yte"]

    model = model_builder(**builder_kwargs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(cfg.LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    n_trainable = count_trainable_params(model)

    start = time.time()
    history = model.fit(
        Xtr, ytr,
        validation_data=(Xva, yva),
        epochs=cfg.EPOCHS,
        batch_size=cfg.BATCH_SIZE,
        verbose=0,
    )
    end = time.time()

    val_prob = model.predict(Xva, verbose=0, batch_size=cfg.BATCH_SIZE).ravel()
    test_prob = model.predict(Xte, verbose=0, batch_size=cfg.BATCH_SIZE).ravel()

    record = {
        "tier": tag,
        "description": description,
        "data_source": data["source"],
        "img_size": cfg.IMG_SIZE,
        "n_train": len(ytr),
        "n_val": len(yva),
        "n_test": len(yte),
        "trainable_params": int(n_trainable),
        "wall_clock_seconds_total": end - start,
        "wall_clock_seconds_per_epoch": (end - start) / cfg.EPOCHS,
        "final_train_loss": float(history.history["loss"][-1]),
        "final_val_loss": float(history.history["val_loss"][-1]),
        "val_metrics": evaluate(yva, val_prob),
        "test_metrics": evaluate(yte, test_prob),
    }
    record.update({k: v for k, v in builder_kwargs.items() if k in ["augment"]})
    save_result(tag, record, cfg.RESULTS_DIR)
    return record


## 6. Tier 1 (sklearn Logistic Regression)

Tier 1 uses the frozen backbone to extract features, then fits a logistic regression classifier on those features. This is a cheap baseline with no backpropagation through the CNN.

In [ ]:
def run_tier1(data, cfg):
    """Extract frozen CNN features and fit a logistic-regression classifier (Tier 1)."""
    logger.info("Running Tier 1")
    Xtr, ytr = data["Xtr"], data["ytr"]
    Xva, yva = data["Xva"], data["yva"]
    Xte, yte = data["Xte"], data["yte"]

    backbone = build_frozen_backbone(img_size=cfg.IMG_SIZE, seed=cfg.SEED)
    n_forward = count_nontrainable_params(backbone)

    start = time.time()
    Ftr = backbone.predict(Xtr, verbose=0, batch_size=cfg.BATCH_SIZE)
    Fva = backbone.predict(Xva, verbose=0, batch_size=cfg.BATCH_SIZE)
    Fte = backbone.predict(Xte, verbose=0, batch_size=cfg.BATCH_SIZE)
    feat_time = time.time()

    clf = LogisticRegression(max_iter=2000, random_state=cfg.SEED)
    clf.fit(Ftr, ytr)
    end = time.time()

    record = {
        "tier": "tier1_frozen_logreg",
        "description": "Frozen random-CNN embeddings + LogisticRegression",
        "data_source": data["source"],
        "img_size": cfg.IMG_SIZE,
        "n_train": len(ytr),
        "n_val": len(yva),
        "n_test": len(yte),
        "trainable_params": int(Ftr.shape[1] + 1),
        "nontrainable_forward_params": int(n_forward),
        "wall_clock_seconds_total": end - start,
        "wall_clock_seconds_feature_extraction": feat_time - start,
        "wall_clock_seconds_logreg_fit": end - feat_time,
        "val_metrics": evaluate(yva, clf.predict_proba(Fva)[:, 1]),
        "test_metrics": evaluate(yte, clf.predict_proba(Fte)[:, 1]),
    }
    save_result("tier1", record, cfg.RESULTS_DIR)
    return record


## 7. Figure Generation

We generate three figures:
1. Confusion matrices for all four tiers.
2. Compute cost vs. F1: comparing trainable parameters and wall-clock time against test F1.
3. Sample images for the reader to enjoy, adding legibility.

A summary table is saved as CSV and Markdown, including the Tier 2 augmentation sensitivity comparison.

In [ ]:
def generate_figures(cfg):
    logger.info("Generating figures")

    def load(tag):
        with open(os.path.join(cfg.RESULTS_DIR, f"{tag}.json")) as f:
            return json.load(f)

    tier1 = load("tier1")
    tier2 = load("tier2_no_aug")
    tier2_aug = load("tier2_aug")
    tier3 = load("tier3")
    sens = load("sensitivity_augmentation")

    # Single-run point estimates for all four tiers
    rows = []
    tm1 = tier1["test_metrics"]
    tm3 = tier3["test_metrics"]
    for rec, label in [(tier1, "Tier 1"), (tier2, "Tier 2"),
                        (tier2_aug, "Tier 2b"), (tier3, "Tier 3")]:
        tm = rec["test_metrics"]
        rows.append({
            "tier": label,
            "params": rec["trainable_params"],
            "time_s": round(rec["wall_clock_seconds_total"], 2),
            "F1": f"{tm['f1_binary_positive']:.4f}",
            "Acc": f"{tm['accuracy']:.4f}",
        })
    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(os.path.join(cfg.OUT_DIR, "results_summary.csv"), index=False)

    def _to_markdown_table(df: pd.DataFrame) -> str:
        header = "| " + " | ".join(df.columns) + " |"
        sep = "| " + " | ".join(["---"] * len(df.columns)) + " |"
        rows_md = [
            "| " + " | ".join(str(v) for v in row) + " |"
            for row in df.itertuples(index=False)
        ]
        return "\n".join([header, sep, *rows_md])

    with open(os.path.join(cfg.OUT_DIR, "results_summary.md"), "w", encoding="utf-8") as f:
        f.write(_to_markdown_table(summary_df))
        f.write(
            f"\n\nAugmentation sensitivity (Tier 2 CNN): "
            f"val F1 {sens['f1_val_no_aug']:.3f} (no aug) vs "
            f"{sens['f1_val_with_aug']:.3f} (aug), delta "
            f"{sens['delta_f1_val_aug_minus_noaug']:+.3f}; "
            f"test F1 {sens['f1_test_no_aug']:.3f} (no aug) vs "
            f"{sens['f1_test_with_aug']:.3f} (aug), delta "
            f"{sens['delta_f1_test_aug_minus_noaug']:+.3f}.\n"
        )

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, (title, rec) in zip(
        axes,
        [
            ("Tier 1", tier1),
            ("Tier 2", tier2),
            ("Tier 2b", tier2_aug),
            ("Tier 3", tier3),
        ],
    ):
        cm = np.array(rec["test_metrics"]["confusion_matrix"])
        ax.imshow(cm, cmap="Blues")
        for i, j in np.ndindex(2, 2):
            ax.text(
                j, i, str(cm[i, j]),
                ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black",
            )
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["pred 0", "pred 1"])
        ax.set_yticks([0, 1])
        ax.set_yticklabels(["true 0", "true 1"])
        ax.set_title(f"{title}\nF1={rec['test_metrics']['f1_binary_positive']:.3f}")
    fig.suptitle("Confusion matrices")
    fig.tight_layout()
    fig.savefig(os.path.join(cfg.FIG_DIR, "confusion_matrices.png"), dpi=150)
    plt.close(fig)

    labels = ["Tier 1", "Tier 2", "Tier 3"]
    f1s = [
        tm1["f1_binary_positive"],
        tier2["test_metrics"]["f1_binary_positive"],
        tm3["f1_binary_positive"],
    ]
    params = [
        tier1["trainable_params"],
        tier2["trainable_params"],
        tier3["trainable_params"],
    ]
    times = [
        tier1["wall_clock_seconds_total"],
        tier2["wall_clock_seconds_total"],
        tier3["wall_clock_seconds_total"],
    ]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
    colors = ["#4C72B0", "#DD8452", "#55A868"]
    ax1.bar(labels, params, color=colors, alpha=0.75)
    ax1.set_ylabel("Trainable params")
    ax1.set_yscale("log")
    ax1b = ax1.twinx()
    ax1b.plot(labels, f1s, "ko-")
    ax1b.set_ylabel("Test F1")
    ax1b.set_ylim(0, 1)
    ax1.set_title("Params vs F1")

    ax2.bar(labels, times, color=colors, alpha=0.75)
    ax2.set_ylabel("Wall time (s)")
    ax2b = ax2.twinx()
    ax2b.plot(labels, f1s, "ko-")
    ax2b.set_ylabel("Test F1")
    ax2b.set_ylim(0, 1)
    ax2.set_title("Time vs F1")
    fig.suptitle("Compute vs accuracy")
    fig.tight_layout()
    fig.savefig(os.path.join(cfg.FIG_DIR, "compute_cost_vs_f1.png"), dpi=150)
    plt.close(fig)

    data = get_dataset(cfg, force_rebuild=False)
    Xtr, ytr = data["Xtr"], data["ytr"]
    fig, axes = plt.subplots(2, 6, figsize=(14, 5))
    for row, label, idxs in [
        (0, "protest=1", np.where(ytr == 1)[0][:6]),
        (1, "protest=0", np.where(ytr == 0)[0][:6]),
    ]:
        for ax, idx in zip(axes[row], idxs):
            ax.imshow(Xtr[idx])
            ax.axis("off")
        axes[row, 0].set_ylabel(label, fontsize=11)
    fig.suptitle("Example images")
    fig.tight_layout()
    fig.savefig(os.path.join(cfg.FIG_DIR, "sample_images.png"), dpi=150)
    plt.close(fig)

    logger.info("Figures saved to %s", cfg.FIG_DIR)


## 8. Main Orchestration

The main function loads the dataset, runs all three tiers (including the augmentation sensitivity check), and then generates the figures.

In [ ]:
def main():
    """Run the full pipeline: load data, train all tiers, and generate figures."""
    parser = argparse.ArgumentParser()
    parser.add_argument("--force_rebuild", action="store_true")
    parser.add_argument("--skip_figures", action="store_true")
    args = parser.parse_args(args=[])

    cfg = Config()
    logger.info("Pipeline started")
    logger.info("Working directory: %s", os.path.abspath(cfg.OUT_DIR))

    data = get_dataset(cfg, force_rebuild=args.force_rebuild)

    run_tier1(data, cfg)

    tier2_no_aug = train_and_evaluate(
        build_finetune_cnn, data, cfg,
        tag="tier2_no_aug", description="CNN without augmentation",
        builder_kwargs={"img_size": cfg.IMG_SIZE, "seed": cfg.SEED, "augment": False},
    )
    tier2_aug = train_and_evaluate(
        build_finetune_cnn, data, cfg,
        tag="tier2_aug", description="CNN with augmentation",
        builder_kwargs={"img_size": cfg.IMG_SIZE, "seed": cfg.SEED, "augment": True},
    )

    save_result(
        "sensitivity_augmentation",
        {
            "comparison": "tier2_cnn: without augmentation vs with augmentation",
            "f1_val_no_aug": tier2_no_aug["val_metrics"]["f1_binary_positive"],
            "f1_val_with_aug": tier2_aug["val_metrics"]["f1_binary_positive"],
            "delta_f1_val_aug_minus_noaug": (
                tier2_aug["val_metrics"]["f1_binary_positive"]
                - tier2_no_aug["val_metrics"]["f1_binary_positive"]
            ),
            "f1_test_no_aug": tier2_no_aug["test_metrics"]["f1_binary_positive"],
            "f1_test_with_aug": tier2_aug["test_metrics"]["f1_binary_positive"],
            "delta_f1_test_aug_minus_noaug": (
                tier2_aug["test_metrics"]["f1_binary_positive"]
                - tier2_no_aug["test_metrics"]["f1_binary_positive"]
            ),
        },
        cfg.RESULTS_DIR,
    )

    train_and_evaluate(
        build_vit_scratch,
        data,
        cfg,
        tag="tier3",
        description="ViT from scratch",
        builder_kwargs={
            "img_size": cfg.IMG_SIZE,
            "patch": cfg.VIT_PATCH,
            "dim": cfg.VIT_DIM,
            "heads": cfg.VIT_HEADS,
            "depth": cfg.VIT_DEPTH,
            "mlp": cfg.VIT_MLP,
            "seed": cfg.SEED,
        },
    )

    if not args.skip_figures:
        generate_figures(cfg)

    logger.info("Pipeline finished. Outputs in %s", cfg.OUT_DIR)


if __name__ == "__main__":
    main()
